# Using GEOparse Python Package - Microglia Data in GEO

!pip install GEOparse

In [48]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
import GEOparse
import requests
import gzip
import io

## Create dictionary of compiled GSEs from research

In [6]:
gse_dict = {
            "Transcriptomic profiling of microglia and astrocytes throughout aging": 137028,
            "Time-dependent changes in microglia transcriptional networks following traumatic brain injury": 132809,
            "Temporal tracking of microglia activation in neurodegeneration at single-cell resolution": 103334,
            "Transcriptional profiling of microglia; current state of the art and future perspectives": 135618,
            "Microglial reactivity and neuroinflammation-driven in motivational behaviors are regulated by Orai1 channels": 310141
           }

## GSE 132809 

### Experimental Design Clarification for above CCI Microglia NanoString Dataset
**Study Structure based on the Methods and Results sections**:
- Model: Controlled cortical impact (CCI)
- Subjects: 2-month-old male C57BL/6J mice

**Groups**:
- Sham (n = 4), 2 dpi (n = 4), 14 dpi (n = 4), 60 dpi (n = 3)

**Microglia were**:
- Isolated from brain tissue via enzymatic digestion
- Stained for CD11b and CD45
- Sorted using FACS
- RNA extracted and analyzed using NanoString nCounter (550 inflammation/immunity genes)

In [33]:
gse_132809 = GEOparse.get_GEO(geo="GSE132809", destdir="./")

21-Feb-2026 05:31:10 DEBUG utils - Directory ./ already exists. Skipping.
21-Feb-2026 05:31:10 INFO GEOparse - File already exist: using local version.
21-Feb-2026 05:31:10 INFO GEOparse - Parsing ./GSE132809_family.soft.gz: 
21-Feb-2026 05:31:10 DEBUG GEOparse - DATABASE: GeoMiame
21-Feb-2026 05:31:10 DEBUG GEOparse - SERIES: GSE132809
21-Feb-2026 05:31:10 DEBUG GEOparse - PLATFORM: GPL26788
21-Feb-2026 05:31:10 DEBUG GEOparse - SAMPLE: GSM3892735
21-Feb-2026 05:31:10 DEBUG GEOparse - SAMPLE: GSM3892736
21-Feb-2026 05:31:11 DEBUG GEOparse - SAMPLE: GSM3892737
21-Feb-2026 05:31:11 DEBUG GEOparse - SAMPLE: GSM3892738
21-Feb-2026 05:31:11 DEBUG GEOparse - SAMPLE: GSM3892739
21-Feb-2026 05:31:11 DEBUG GEOparse - SAMPLE: GSM3892740
21-Feb-2026 05:31:11 DEBUG GEOparse - SAMPLE: GSM3892741
21-Feb-2026 05:31:11 DEBUG GEOparse - SAMPLE: GSM3892742
21-Feb-2026 05:31:11 DEBUG GEOparse - SAMPLE: GSM3892743
21-Feb-2026 05:31:11 DEBUG GEOparse - SAMPLE: GSM3892744
21-Feb-2026 05:31:11 DEBUG GEOpars

In [34]:
for i, (gpl_name, gpl) in enumerate(gse_132809.gpls.items()):
    print("Name: ", gpl_name)
    print("Metadata:",)
    for key, value in gpl.metadata.items():
        print(" - %s : %s" % (key, ", ".join(value)))
    print("Table data:",)
    print(gpl.table.head())

Name:  GPL26788
Metadata:
 - title : NanoString nCounter® Mouse Inflammation v2 Panel
 - geo_accession : GPL26788
 - status : Public on Jun 18 2019
 - submission_date : Jun 17 2019
 - last_update_date : Jun 18 2019
 - technology : other
 - distribution : custom-commercial
 - organism : Mus musculus
 - taxid : 10090
 - manufacturer : Nanostring
 - manufacture_protocol : see manufacturer's website
 - contact_name : Michael,,Whalen
 - contact_email : mwhalen@mgh.harvard.edu
 - contact_phone : 617-643-3933
 - contact_laboratory : Acute Brain Injury Research Laboratory
 - contact_department : Department of Pediatrics
 - contact_institute : Massachusetts General Hospital
 - contact_address : 149 13th Street Room 6303
 - contact_city : Charlestown
 - contact_state : MA
 - contact_zip/postal_code : 02129
 - contact_country : USA
 - data_row_count : 547
Table data:
            ID  Code.Class Gene.Name       GB_ACC
0  NM_019552.2  Endogenous    Abcb10  NM_019552.2
1  NM_011076.1  Endogenous    A

In [37]:
for i, (gsm_name, gsm) in enumerate(gse_132809.gsms.items()):
    print("Name: ", gsm_name)
    print("Metadata:",)
    for key, value in gsm.metadata.items():
        print(" - %s : %s" % (key, ", ".join(value)))
    print ("Table data:",)
    print (gsm.table.head(20))
    if i==0:
        break
    

Name:  GSM3892735
Metadata:
 - title : 20141015_cci_01_01.RCC
 - geo_accession : GSM3892735
 - status : Public on Jun 18 2019
 - submission_date : Jun 17 2019
 - last_update_date : Jun 18 2019
 - type : RNA
 - channel_count : 1
 - source_name_ch1 : brain
 - organism_ch1 : Mus musculus
 - taxid_ch1 : 10090
 - characteristics_ch1 : injury: sham
 - treatment_protocol_ch1 : NA
 - growth_protocol_ch1 : NA
 - molecule_ch1 : total RNA
 - extract_protocol_ch1 : Microglia isolated by FACS were stored in RNAlater Stabilization Solution (AM7020, Invitrogen) at -80°C until further processing. Total RNA from the sorted cells was extracted using miRNeasy Micro Kit from Qiagen (Redwood City, CA) for purification of total RNA from small amounts of cells according to the manufacturer's instructions. mRNA expression analysis with the NanoString nCounter® Mouse Inflammation v2 Panel was performed according to the manufacturer's protocol (NanoString Technologies Inc., Seattle, WA).
 - label_ch1 : N/A
 - l

## GSE 137028 

### Data Structure Clarification
This dataset is RNA-seq (“Expression profiling by high throughput sequencing”). Although GEO lists 60 samples, GEOparse does not return per-GSM SOFT expression tables because the processed data were not stored in individual GSM records.

**Instead**:
- Raw sequencing data are stored in SRA (BioProject PRJNA564299; SRP220678).
- The processed expression matrix is provided as a single supplementary file: `GSE137028_gene_fpkm.xls.gz`

**This file contains**:
- 35,999 genes
- 20 biological samples (columns)
- FPKM-normalized expression values

The 60 samples listed in GEO likely correspond to raw sequencing runs or technical libraries, which were aggregated into 20 condition-level profiles in the processed matrix.

**Factorial Column names**:
- Genotype: WT vs AD
- Age: 2m, 4m, 6m, 9m, 12m
- Group (likely sex): M and A

So there are **2×5×2=20 processed samples**

Thus, GEOparse shows no GSM tables because the usable expression data are stored in the supplementary file rather than in SOFT-formatted per-sample tables.

In [38]:
gse_137028 = GEOparse.get_GEO(geo="GSE137028", destdir="./")

21-Feb-2026 05:56:34 DEBUG utils - Directory ./ already exists. Skipping.
21-Feb-2026 05:56:34 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE137nnn/GSE137028/soft/GSE137028_family.soft.gz to ./GSE137028_family.soft.gz
100%|██████████| 5.77k/5.77k [00:00<00:00, 18.6kB/s]
21-Feb-2026 05:56:36 DEBUG downloader - Size validation passed
21-Feb-2026 05:56:36 DEBUG downloader - Moving /tmp/tmp14accjkl to /home/jovyan/GSE137028_family.soft.gz
21-Feb-2026 05:56:36 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE137nnn/GSE137028/soft/GSE137028_family.soft.gz
21-Feb-2026 05:56:36 INFO GEOparse - Parsing ./GSE137028_family.soft.gz: 
21-Feb-2026 05:56:36 DEBUG GEOparse - DATABASE: GeoMiame
21-Feb-2026 05:56:36 DEBUG GEOparse - SERIES: GSE137028
21-Feb-2026 05:56:36 DEBUG GEOparse - PLATFORM: GPL24247
21-Feb-2026 05:56:36 DEBUG GEOparse - SAMPLE: GSM4065891
21-Feb-2026 05:56:36 DEBUG GEOparse - SAMPLE: GSM4065892
21-Feb-2026 05:56:36 DEBUG G

In [40]:
for i, (gpl_name, gpl) in enumerate(gse_137028.gpls.items()):
    print("Name: ", gpl_name)
    print("Metadata:",)
    for key, value in gpl.metadata.items():
        print(" - %s : %s" % (key, ", ".join(value)))
    print("Table data:",)
    print(gpl.table.head())

Name:  GPL24247
Metadata:
 - title : Illumina NovaSeq 6000 (Mus musculus)
 - geo_accession : GPL24247
 - status : Public on Nov 08 2017
 - submission_date : Nov 08 2017
 - last_update_date : Mar 08 2019
 - technology : high-throughput sequencing
 - distribution : virtual
 - organism : Mus musculus
 - taxid : 10090
 - contact_name : ,,GEO
 - contact_country : USA
 - data_row_count : 0
Table data:
Empty DataFrame
Columns: []
Index: []


In [44]:
for i, (gsm_name, gsm) in enumerate(gse_137028.gsms.items()):
    print("Name: ", gsm_name)
    print("Metadata:",)
    for key, value in gsm.metadata.items():
        print(" - %s : %s" % (key, ", ".join(value)))
    print ("Table data:",)
    print (gsm.table.head(20))
    if i==0:
        break
    

Name:  GSM4065891
Metadata:
 - title : WT2m1M
 - geo_accession : GSM4065891
 - status : Public on Apr 17 2020
 - submission_date : Sep 06 2019
 - last_update_date : Apr 17 2020
 - type : SRA
 - channel_count : 1
 - source_name_ch1 : Brain
 - organism_ch1 : Mus musculus
 - taxid_ch1 : 10090
 - characteristics_ch1 : strain: C57BL/6, age: 2-month-old
 - molecule_ch1 : total RNA
 - extract_protocol_ch1 : RNA was isolated from flow cytometry-sorted cell populations using RNeasy Micro Kit (74004, Qiagen), A total amount of 10 ng RNA per sample was used as input material for the RNA sample preparations. Libraries were generated using SMART-Seq v4 Ultra Low Input RNA Kit (634892, Takara Bio USA, Mountain View, CA, USA) following manufacturer’s recommendations and index codes were added to attribute sequences to each sample. Briefly, first-strand cDNA synthesis from total RNA is primed by the 3’ SMART-Seq CDS Primer II A and uses the SMART-Seq v4 Oligonucleotide for template switching at the 5’

In [45]:
print(gse_137028.metadata.get("type", []))
print(gse_137028.metadata.get("supplementary_file", [])[:10])
print(gse_137028.relations)

['Expression profiling by high throughput sequencing']
['ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE137nnn/GSE137028/suppl/GSE137028_gene_fpkm.xls.gz']
{'BioProject': ['https://www.ncbi.nlm.nih.gov/bioproject/PRJNA564299'], 'SRA': ['https://www.ncbi.nlm.nih.gov/sra?term=SRP220678']}


In [49]:
url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE137nnn/GSE137028/suppl/GSE137028_gene_fpkm.xls.gz"
r = requests.get(url)
compressed_file = io.BytesIO(r.content)
with gzip.open(compressed_file) as f:
    df_137028 = pd.read_excel(f)

print(df_137028.head())
print(df_137028.shape)

              gene_id         WT2mM         WT4mM         WT6mM         WT9mM  \
0  ENSMUSG00000027447  68165.002270  77992.193419  66575.831713  43885.059333   
1  ENSMUSG00000024661   3845.103176   3692.195390   4967.818359   4061.208478   
2  ENSMUSG00000038642  12817.844190  13731.290151  19752.756756  11134.137593   
3  ENSMUSG00000049775  10578.664606  10245.166862  12059.352638   8408.865137   
4  ENSMUSG00000036905  10487.262630  11009.635127  13231.701913   7128.882128   

         WT12mM         AD2mM         AD4mM         AD6mM         AD9mM  ...  \
0  39332.275233  76493.920895  51811.192955  64800.386824  38448.689027  ...   
1   3834.502813   4023.522699   3927.298858   4896.643218   3907.164574  ...   
2   9986.886380  13577.022817  11830.694712  18414.985153   8232.700179  ...   
3   6129.744878  10192.700050   9137.571661  13293.544775   6169.016818  ...   
4   7616.056069  12719.295809  10962.208895  13423.054268   5844.781359  ...   

          WT2mA         WT4mA   

## GSE 103334